# 以DataFrame表格形式呈现不同模型给出的正向情感得分

In [1]:
# 运行准备：按示例代码5.1～5.2读取并划分英文影评数据，并使用示例代码5.51加载分词器
import os
import csv
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast

data_file = "../pybook-data/ch5/imdb_labelled.txt"
df = pd.read_csv(data_file, names=["sentence", "label"], sep="\t", quoting=csv.QUOTE_NONE)
sents = df["sentence"].values
y = df["label"].values

out_dir = "output"
train_file = f"{out_dir}/imdb_labelled_train.csv"
test_file = f"{out_dir}/imdb_labelled_test.csv"
if os.path.exists(train_file) and os.path.exists(test_file):
    print(f'Files exist at "{train_file}" and "{test_file}"')
    df_train, df_test = pd.read_csv(train_file), pd.read_csv(test_file)
    sents_train, sents_test = df_train["sentence"].values, df_test["sentence"].values
    y_train, y_test = df_train["label"].values, df_test["label"].values
else:
    os.makedirs(out_dir, exist_ok=True)
    sents_train, sents_test, y_train, y_test = train_test_split(sents, y, test_size=0.2, random_state=1)
    df_train = pd.DataFrame({"sentence": sents_train, "label": y_train})
    df_test = pd.DataFrame({"sentence": sents_test, "label": y_test})
    df_train.to_csv(train_file, index=False)
    df_test.to_csv(test_file, index=False)

from transformers import BertTokenizerFast, BertForSequenceClassification, get_linear_schedule_with_warmup
from tqdm import tqdm
import time
import numpy as np

tokenizer = BertTokenizerFast.from_pretrained('./bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('./bert-base-uncased', num_labels=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/Users/xinzijie/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Files exist at "output/imdb_labelled_train.csv" and "output/imdb_labelled_test.csv"


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1994.15it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: ./bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkp

## 定义面向BERT模型的数据集类BertDataset

In [2]:
class BertDataset(Dataset):
    def __init__(self, sents, labels, tokenizer, max_len=64):
        self.texts = sents
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            add_special_tokens=True,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoded.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

## 创建BertDataset和DataLoader对象，用于加载训练数据

In [3]:
train_ds = BertDataset(sents_train, y_train, tokenizer)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

## 为微调BERT设定训练选项

In [4]:
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
max_epoch = 10
total_steps = len(train_loader) * max_epoch
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

## 微调BERT模型

In [ ]:
t0 = time.perf_counter()
for epoch in range(max_epoch):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")
if device.type == "cuda":
    torch.cuda.synchronize(device)
total_time = time.perf_counter() - t0
print(f"Training finished in {total_time/60:.2f} min")

Epoch 1:   8%|▊         | 2/25 [02:36<27:47, 72.51s/it] 

## BERT性能评测

In [ ]:
from sklearn.metrics import classification_report

test_ds = BertDataset(sents_test, [0]*len(sents_test), tokenizer) #推理阶段无真实标签可用
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)
model.eval()
all_preds = []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
print(classification_report(y_test, all_preds, digits=3))

## 用示例代码5.49的样本测试微调后的BERT模型

In [ ]:
input_texts = ['this is a good movie', 'the movie is horrible', 'a horrible movie', 'not a bad movie']
input_ds = BertDataset(input_texts, [0]*len(input_texts), tokenizer)
input_loader = DataLoader(input_ds, batch_size=16, shuffle=False)
model.eval()
all_scores = []
with torch.no_grad():
    for batch in input_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        scores = torch.softmax(outputs.logits, dim=1).cpu().numpy()
        all_scores.append(scores)
scores_bert = np.vstack(all_scores)
print(scores_bert)